# Depth Sweep Results — plain nets vs ResNet
Loads everything `train_depth_sweep.py` wrote to `runs/`:

1. **Load a model** (any checkpoint) and run a forward/backward pass on val images
2. **Correct / incorrect** validation images, visualized
3. **Accuracy curves** across all models, top-1 and top-5
4. Bonus: optimizer path length from the saved checkpoints

In [ ]:
import json, math, re
from pathlib import Path

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision.models.resnet import ResNet, BasicBlock

RUNS = Path("runs")                       # <-- output dir of train_depth_sweep.py
DATA = Path("/home/stephen/imagenet")     # <-- edit
device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_DEPTHS = {"plain8": 8, "plain14": 14, "plain20": 20, "plain26": 26,
                "plain34": 34, "plain56": 56, "plain74": 74, "resnet74": 74}
MODELS = [m for m in MODEL_DEPTHS if (RUNS / m).exists()]

def color_of(name):
    """Plain nets: dark->light with depth. ResNet: red."""
    if name == "resnet74":
        return "crimson"
    depths = sorted(d for m, d in MODEL_DEPTHS.items() if m != "resnet74")
    return plt.cm.viridis(depths.index(MODEL_DEPTHS[name]) / (len(depths) - 1))

print(torch.__version__, device, "| runs found:", MODELS)

In [ ]:
# --- model definitions (mirrors train_depth_sweep.py) ---
class PlainBasicBlock(BasicBlock):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.downsample = None

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        return out

LAYER_CFG = {8: [1,1,1,1], 14: [2,1,1,2], 20: [2,2,3,2], 26: [3,3,3,3],
             34: [4,4,4,4], 56: [3,4,17,3], 74: [3,4,26,3]}

def make_net(depth_target, use_skip, num_classes=1000):
    block = BasicBlock if use_skip else PlainBasicBlock
    model = ResNet(block, LAYER_CFG[depth_target], num_classes=num_classes)
    if depth_target == 8:
        model.layer4 = nn.Identity()
        model.fc = nn.Linear(256, num_classes)
    return model

def blocks_of(model):
    return [b for s in (model.layer1, model.layer2, model.layer3, model.layer4)
            if isinstance(s, nn.Sequential) for b in s]

In [ ]:
# --- metrics loading: prefer metrics.npz, fall back to metrics.jsonl ---
def load_metrics(name):
    d = RUNS / name
    if (d / "metrics.npz").exists():
        return dict(np.load(d / "metrics.npz"))
    recs = [json.loads(l) for l in open(d / "metrics.jsonl") if l.strip()]
    out = {}
    for k in recs[0]:
        if k in ("block_grad_norms", "block_acts"):
            continue
        out[k] = np.array([r.get(k, np.nan) for r in recs], dtype=np.float64)
    out["block_grad_norms"] = np.array([r["block_grad_norms"] for r in recs])
    return out

M = {name: load_metrics(name) for name in MODELS}
for name in MODELS:
    print(f"{name:9s} records={len(M[name]['step']):5d} "
          f"last step={int(M[name]['step'][-1])}")

## 1. Load a model and run a forward/backward pass

`load_model(name)` gives you the final weights; `load_model(name, step=20_000)`
gives any intermediate checkpoint (they exist every 1000 steps).

In [ ]:
def list_ckpts(name):
    return sorted(int(re.search(r"(\d+)", p.stem).group(1))
                  for p in (RUNS / name).glob("ckpt_step*.pt"))

def load_model(name, step=None):
    d = RUNS / name
    path = d / "final.pt" if step is None else d / f"ckpt_step{step:06d}.pt"
    model = make_net(MODEL_DEPTHS[name], use_skip=name.startswith("resnet"))
    model.load_state_dict(torch.load(path, map_location=device))
    return model.to(device).eval()

print("plain74 checkpoints:", list_ckpts("plain74")[:5], "...",
      list_ckpts("plain74")[-2:])

In [ ]:
# --- validation data (same transforms as training script) ---
import torchvision.datasets as dsets, torchvision.transforms as T

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
val_ds = dsets.ImageFolder(DATA / "ILSVRC/Data/CLS-LOC/val",
    T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(),
               T.Normalize(MEAN, STD)]))
val_loader = torch.utils.data.DataLoader(val_ds, 64, shuffle=True,
    num_workers=4, pin_memory=True)

# wnid -> human-readable name (Kaggle CLS-LOC ships this mapping)
wnid_words = {}
mapping = DATA / "LOC_synset_mapping.txt"
if mapping.exists():
    for line in open(mapping):
        wnid, words = line.strip().split(" ", 1)
        wnid_words[wnid] = words.split(",")[0]

def class_name(idx):
    if 0 <= idx < len(val_ds.classes):
        wnid = val_ds.classes[idx]
        return wnid_words.get(wnid, wnid)
    return f"cls{idx}"

def denorm(x):  # CHW tensor -> HWC numpy in [0,1] for imshow
    img = x.cpu() * torch.tensor(STD).view(3,1,1) + torch.tensor(MEAN).view(3,1,1)
    return img.clamp(0, 1).permute(1, 2, 0).numpy()

print(len(val_ds), "val images,", len(val_ds.classes), "classes")

In [ ]:
# --- forward + backward on a batch of val images ---
name = "plain74" if "plain74" in MODELS else MODELS[-1]
model = load_model(name)

x, y = next(iter(val_loader))
x, y = x[:16].to(device), y[:16].to(device)

logits = model(x)
loss = F.cross_entropy(logits, y)
model.zero_grad(set_to_none=True)
loss.backward()

probs = logits.softmax(1)
top5p, top5i = probs.topk(5, dim=1)
print(f"{name}: batch loss={loss.item():.3f}, "
      f"batch top-1={(logits.argmax(1) == y).float().mean().item():.3f}\n")
for i in range(3):
    guesses = ", ".join(f"{class_name(top5i[i,j].item())} {top5p[i,j]:.2f}"
                        for j in range(5))
    print(f"true: {class_name(y[i].item()):22s} -> {guesses}")

In [ ]:
# --- per-block gradient norms from that backward pass ---
def block_grad_profile(model):
    return [math.sqrt(sum(float(p.grad.norm())**2 for p in b.parameters()
                          if p.grad is not None)) for b in blocks_of(model)]

fig, ax = plt.subplots(figsize=(9, 3.5))
prof = block_grad_profile(model)
ax.bar(range(len(prof)), prof, color=color_of(name), label=name)

# same batch through the resnet for contrast, if it exists
if name != "resnet74" and "resnet74" in MODELS:
    rn = load_model("resnet74")
    rl = F.cross_entropy(rn(x), y)
    rn.zero_grad(set_to_none=True); rl.backward()
    rprof = block_grad_profile(rn)
    ax.plot(range(len(rprof)), rprof, "o-", color="crimson",
            label="resnet74", ms=4)
    del rn

ax.set(xlabel="block index (input → output)", ylabel="grad L2 norm",
       title="per-block gradient norms, one val batch")
ax.legend(); plt.tight_layout()

## 2. Correct / incorrect validation images

In [ ]:
@torch.no_grad()
def scan_val(model, n_images=2000):
    """Collect predictions over the first n_images of a shuffled val pass."""
    idxs, ys, preds, confs = [], [], [], []
    seen = 0
    for x, y in val_loader:
        x = x.to(device, non_blocking=True)
        with torch.amp.autocast(device, enabled=(device == "cuda")):
            p = model(x).softmax(1)
        conf, pred = p.max(1)
        ys.append(y); preds.append(pred.cpu()); confs.append(conf.float().cpu())
        idxs.append(x.cpu())      # keep images for display
        seen += y.numel()
        if seen >= n_images:
            break
    return (torch.cat(idxs), torch.cat(ys), torch.cat(preds),
            torch.cat(confs))

name = "resnet74" if "resnet74" in MODELS else MODELS[-1]
model = load_model(name)
imgs, ys, preds, confs = scan_val(model, n_images=2000)
correct = preds == ys
print(f"{name}: scanned {len(ys)} images, top-1 = {correct.float().mean():.4f}")

In [ ]:
def show_grid(mask, title, n=8, order=None):
    idx = torch.nonzero(mask).flatten()
    if order is not None:                     # e.g. sort by confidence
        idx = idx[torch.argsort(order[idx], descending=True)]
    idx = idx[:n]
    fig, axes = plt.subplots(2, 4, figsize=(13, 7))
    for ax, i in zip(axes.flat, idx):
        i = i.item()
        ax.imshow(denorm(imgs[i]))
        ok = preds[i] == ys[i]
        ax.set_title(f"true: {class_name(ys[i].item())}\n"
                     f"pred: {class_name(preds[i].item())} ({confs[i]:.2f})",
                     fontsize=9, color="green" if ok else "firebrick")
        ax.axis("off")
    fig.suptitle(f"{name} — {title}", fontsize=13)
    plt.tight_layout()

show_grid(correct, "correct, most confident", order=confs)

In [ ]:
show_grid(~correct, "wrong, most confident (the interesting failures)",
          order=confs)

In [ ]:
show_grid(~correct, "wrong, least confident (genuinely unsure)",
          order=-confs)

## 3. Accuracy curves — all models

Quick-val (fixed 5k subset, every 100 steps) for the fine-grained story;
full 50k val (every 1000 steps) for the honest numbers.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for m in MODELS:
    ax.plot(M[m]["step"], M[m]["val_top1"], color=color_of(m), label=m,
            lw=1.4, alpha=0.9)
ax.set(xlabel="step", ylabel="top-1 accuracy",
       title="quick val (5k subset), every 100 steps")
ax.legend(ncol=2, fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
for m in MODELS:
    steps = M[m]["step"]
    for ax, key, lbl in [(axes[0], "full_val_top1", "top-1"),
                          (axes[1], "full_val_top5", "top-5")]:
        v = M[m][key]
        keep = ~np.isnan(v)
        ax.plot(steps[keep], v[keep], color=color_of(m), label=m, lw=1.6)
for ax, lbl in zip(axes, ["top-1", "top-5"]):
    ax.set(xlabel="step", ylabel=f"{lbl} accuracy",
           title=f"full 50k val — {lbl}")
    ax.grid(alpha=0.3)
axes[0].legend(ncol=2, fontsize=9)
plt.tight_layout()

In [ ]:
# --- He et al.-style: final error vs depth ---
rows = []
for m in MODELS:
    v1, v5 = M[m]["full_val_top1"], M[m]["full_val_top5"]
    f1 = v1[~np.isnan(v1)][-1]
    f5 = v5[~np.isnan(v5)][-1]
    rows.append((m, MODEL_DEPTHS[m], f1, f5))
    print(f"{m:9s} depth={MODEL_DEPTHS[m]:2d}  "
          f"top-1={f1:.4f}  top-5={f5:.4f}  "
          f"err-1={100*(1-f1):.2f}%")

plain = [(d, 1 - a1) for m, d, a1, _ in rows if m.startswith("plain")]
fig, ax = plt.subplots(figsize=(7, 4.5))
if plain:
    ds, es = zip(*sorted(plain))
    ax.plot(ds, [100 * e for e in es], "o-", color="steelblue",
            label="plain (no skip)")
for m, d, a1, _ in rows:
    if m == "resnet74":
        ax.plot([d], [100 * (1 - a1)], "*", color="crimson", ms=16,
                label="resnet74 (skip)")
ax.set(xlabel="depth (weighted layers)", ylabel="final top-1 error (%)",
       title="the degradation problem")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()

In [ ]:
# --- train loss, for the "degradation is a *training* problem" argument ---
fig, ax = plt.subplots(figsize=(11, 5))
for m in MODELS:
    ax.plot(M[m]["step"], M[m]["train_loss_avg"], color=color_of(m),
            label=m, lw=1.2, alpha=0.9)
ax.set(xlabel="step", ylabel="train loss (100-step mean)",
       title="training loss")
ax.legend(ncol=2, fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()

## 4. Bonus: optimizer path length from checkpoints

Distance traveled in parameter space between consecutive 1000-step
checkpoints — connects to the loss-landscape visualizations.

In [ ]:
@torch.no_grad()
def ckpt_distances(name):
    steps = list_ckpts(name)
    dists, prev = [], None
    for s in steps:
        sd = torch.load(RUNS / name / f"ckpt_step{s:06d}.pt",
                        map_location="cpu")
        flat = torch.cat([v.flatten().float() for v in sd.values()])
        if prev is not None:
            dists.append((s, (flat - prev).norm().item()))
        prev = flat
    return dists

fig, ax = plt.subplots(figsize=(10, 4.5))
for m in MODELS:                 # heavy I/O: trim this list if impatient
    d = ckpt_distances(m)
    if d:
        s, v = zip(*d)
        ax.plot(s, v, color=color_of(m), label=m, lw=1.3)
ax.set(xlabel="step", ylabel="‖θₜ − θₜ₋₁‖",
       title="parameter distance per 1000 steps")
ax.legend(ncol=2, fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()